# Portfolio Optimization with Monte Carlo Simulation

This notebook demonstrates interactive portfolio optimization using Geometric Brownian Motion with quasi-Monte Carlo simulation.

## Setup

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add src to path
sys.path.insert(0, '../src')

from data_processing import DataLoader
from models import GeometricBrownianMotion, VaRCalculator
from simulation import MonteCarloSimulator, EventSimulator
from optimization import PortfolioOptimizer
from utils import load_config, calculate_log_returns

%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## 1. Load Configuration and Data

In [ ]:
# Load config
config = load_config('../config/config.yaml')

# Load data
data_loader = DataLoader(
    excel_path='../data/raw/Book1__1_.xlsx',
    clean_data_sheet='Clean Data',
    stock_data_sheet='Stock Data'
)

data = data_loader.load_all_data()
stock_prices = data['stock_prices']
indicators = data['indicators']

print(f"Data loaded: {len(stock_prices)} days, {len(stock_prices.columns)} stocks")
print(f"Date range: {stock_prices.index[0]} to {stock_prices.index[-1]}")

In [ ]:
# Display sample data
stock_prices.head()

In [ ]:
# Calculate returns
stock_returns = calculate_log_returns(stock_prices)
stock_returns.head()

## 2. Exploratory Data Analysis

In [ ]:
# Plot stock price evolution
fig, ax = plt.subplots(figsize=(14, 6))
normalized_prices = stock_prices / stock_prices.iloc[0]
normalized_prices.plot(ax=ax, alpha=0.5, legend=False)
ax.set_title('Normalized Stock Prices')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Price')
plt.tight_layout()
plt.show()

In [ ]:
# Return statistics
return_stats = pd.DataFrame({
    'Mean': stock_returns.mean() * 252,
    'Volatility': stock_returns.std() * np.sqrt(252),
    'Sharpe': (stock_returns.mean() * 252 - 0.07) / (stock_returns.std() * np.sqrt(252))
})
return_stats.sort_values('Sharpe', ascending=False).head(10)

## 3. Initialize GBM Model

In [ ]:
# Initialize GBM
gbm = GeometricBrownianMotion(
    stock_returns=stock_returns,
    external_factors=indicators,
    estimation_window=252
)

# Estimate parameters
params = gbm.estimate_parameters()

print(f"\nAverage annual return: {params['mu'].mean():.2%}")
print(f"Average annual volatility: {params['sigma'].mean():.2%}")

In [ ]:
# Visualize correlation structure
plt.figure(figsize=(12, 10))
sns.heatmap(params['correlation'], cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Stock Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Run Monte Carlo Simulation

In [ ]:
# Initialize event simulator
event_simulator = EventSimulator(indicators, 'RBI_Repo_Change')

# Initialize MC simulator
simulator = MonteCarloSimulator(gbm, event_simulator)

# Initial prices
S0 = stock_prices.iloc[-1].values

# Run simulation (smaller scale for notebook)
print("Running simulation...")
result = simulator.run_single_simulation(
    S0=S0,
    T=252,
    n_paths=1000,
    use_sobol=True,
    include_events=True,
    random_state=42
)

print(f"\nSimulation complete: {result['paths'].shape[0]} paths")

In [ ]:
# Visualize simulated paths for first stock
plt.figure(figsize=(12, 6))

for i in range(100):
    plt.plot(result['paths'][i, :, 0], alpha=0.1, color='blue')

mean_path = result['paths'][:, :, 0].mean(axis=0)
plt.plot(mean_path, color='red', linewidth=2, label='Mean Path')

plt.title(f'{stock_prices.columns[0]} - Simulated Price Paths')
plt.xlabel('Time (days)')
plt.ylabel('Price')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Portfolio Optimization

In [ ]:
# Estimate parameters from simulation
simulated_returns = result['returns']
path_avg_returns = simulated_returns.mean(axis=1)

expected_returns = path_avg_returns.mean(axis=0) * 252
covariance_matrix = np.cov(path_avg_returns.T) * 252

# Initialize optimizer
optimizer = PortfolioOptimizer(
    expected_returns=expected_returns,
    covariance_matrix=covariance_matrix,
    risk_free_rate=0.07
)

In [ ]:
# Optimize for max Sharpe
optimal_portfolio = optimizer.optimize_max_sharpe(min_weight=0.0, max_weight=0.4)

print("\nOptimal Portfolio:")
print(f"Expected Return: {optimal_portfolio['return']:.2%}")
print(f"Volatility: {optimal_portfolio['volatility']:.2%}")
print(f"Sharpe Ratio: {optimal_portfolio['sharpe_ratio']:.4f}")

In [ ]:
# Display top allocations
weights_df = pd.DataFrame({
    'Stock': stock_prices.columns,
    'Weight': optimal_portfolio['weights']
})
weights_df = weights_df[weights_df['Weight'] > 0.001].sort_values('Weight', ascending=False)

plt.figure(figsize=(12, 6))
plt.barh(range(len(weights_df)), weights_df['Weight'] * 100)
plt.yticks(range(len(weights_df)), weights_df['Stock'])
plt.xlabel('Weight (%)')
plt.title('Optimal Portfolio Allocations')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Efficient Frontier

In [ ]:
# Calculate efficient frontier
print("Calculating efficient frontier...")
frontier = optimizer.efficient_frontier(n_points=30, min_weight=0.0, max_weight=0.4)

# Plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(frontier['volatility'], frontier['return'],
                     c=frontier['sharpe_ratio'], cmap='viridis', s=50)
plt.scatter(optimal_portfolio['volatility'], optimal_portfolio['return'],
           color='red', s=200, marker='*', label='Optimal Portfolio', zorder=5)
plt.colorbar(scatter, label='Sharpe Ratio')
plt.xlabel('Volatility (Annual)')
plt.ylabel('Expected Return (Annual)')
plt.title('Efficient Frontier')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Value at Risk Analysis

In [ ]:
# Calculate VaR
var_calc = VaRCalculator([0.90, 0.95, 0.99])

var_results = var_calc.calculate_var_from_paths(
    price_paths=result['paths'],
    weights=optimal_portfolio['weights'],
    initial_value=1.0,
    method='historical'
)

# Display VaR summary
var_summary = var_calc.var_summary(
    portfolio_returns=var_results['mean_return'],
    method='historical'
)

print("\nValue at Risk Summary:")
print(var_summary)

In [ ]:
# Plot return distribution
portfolio_values = (result['final_prices'] * optimal_portfolio['weights']).sum(axis=1)
terminal_returns = (portfolio_values - 1.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(terminal_returns, bins=50, alpha=0.7, edgecolor='black')
axes[0].axvline(terminal_returns.mean(), color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {terminal_returns.mean():.2%}')
axes[0].axvline(np.percentile(terminal_returns, 5), color='orange',
               linestyle='--', linewidth=2, label='VaR 95%')
axes[0].set_xlabel('Portfolio Return')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Terminal Return Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative distribution
sorted_returns = np.sort(terminal_returns)
cumulative = np.arange(1, len(sorted_returns) + 1) / len(sorted_returns)
axes[1].plot(sorted_returns, cumulative, linewidth=2)
axes[1].axhline(0.05, color='orange', linestyle='--', label='5th Percentile')
axes[1].axhline(0.95, color='green', linestyle='--', label='95th Percentile')
axes[1].set_xlabel('Portfolio Return')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_title('Cumulative Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Compare Different Strategies

In [ ]:
# Equal weight portfolio
equal_weights = np.ones(len(stock_prices.columns)) / len(stock_prices.columns)

# Min variance portfolio
min_var = optimizer.optimize_mean_variance(target_return=None, min_weight=0.0, max_weight=0.4)

# Risk parity
risk_parity = optimizer.optimize_risk_parity()

# Compare
comparison = pd.DataFrame({
    'Equal Weight': [
        equal_weights @ expected_returns,
        np.sqrt(equal_weights @ covariance_matrix @ equal_weights),
        (equal_weights @ expected_returns - 0.07) / np.sqrt(equal_weights @ covariance_matrix @ equal_weights)
    ],
    'Min Variance': [
        min_var['return'],
        min_var['volatility'],
        min_var['sharpe_ratio']
    ],
    'Risk Parity': [
        risk_parity['return'],
        risk_parity['volatility'],
        risk_parity['sharpe_ratio']
    ],
    'Max Sharpe': [
        optimal_portfolio['return'],
        optimal_portfolio['volatility'],
        optimal_portfolio['sharpe_ratio']
    ]
}, index=['Return', 'Volatility', 'Sharpe Ratio'])

print("\nStrategy Comparison:")
print(comparison)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

comparison.T.plot(kind='bar', ax=axes[0], y='Return', legend=False)
axes[0].set_title('Expected Return')
axes[0].set_ylabel('Annual Return')
axes[0].tick_params(axis='x', rotation=45)

comparison.T.plot(kind='bar', ax=axes[1], y='Volatility', legend=False, color='orange')
axes[1].set_title('Volatility')
axes[1].set_ylabel('Annual Volatility')
axes[1].tick_params(axis='x', rotation=45)

comparison.T.plot(kind='bar', ax=axes[2], y='Sharpe Ratio', legend=False, color='green')
axes[2].set_title('Sharpe Ratio')
axes[2].set_ylabel('Sharpe Ratio')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()